In [ ]:
#@title Installing required packages

!pip install deepface
!pip install retina-face

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.4/88.4 kB 12.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 11.0 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.6.0-py2.py3-none-any.whl size=117029 sha256=7ae68e00b634b1255763a9b30e79b92fed57f51494310284573bd267830f2d09
  Stored in directory: /root/.cache/pip/wheels/d6/6d/5d/5b73fa0f46d01a793713f8859201361e9e581ced8c75e5c6a3
Successfully built fire


In [ ]:
#@title Import Packages

# Data handling
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Image specific
import cv2
from google.colab.patches import cv2_imshow

# RetinaFace
from retinaface import RetinaFace
from retinaface.commons import preprocess, postprocess
# DeepFace
from deepface import DeepFace

import uuid
from tqdm.notebook import tqdm

import os
from os import path
import warnings
import pickle

24-04-18 08:52:44 - Directory /root/.deepface created
24-04-18 08:52:44 - Directory /root/.deepface/weights created


In [ ]:
#@title Load an unzip images for face detection

!unzip -q ../data/posts.zip -d ./images
!unzip -q ../data/2024-01-03-database-final-v4.zip -d ./database_final-v1
!unzip -q ../data/2024-04-17-Faces-Posts-Images.zip -d ./Exports

In [ ]:
!unzip -q ../data/2024-04-17-Faces-Posts-Images.zip

In [ ]:
#@title Import data farmes

all_faces_v2_df = pd.read_csv('../data/2024-03-18-Retina-Face-Posts-Detection.csv')
#images_v2_df = pd.read_csv('../dataimages_v2.csv')
#all_faces_tasks_df = pd.read_csv('../dataAllFaces-Tasks.csv')

In [ ]:
import ast

all_faces_v2_df["facial_area"] = all_faces_v2_df["facial_area"].apply(ast.literal_eval)
all_faces_v2_df["right_eye"] = all_faces_v2_df["right_eye"].apply(ast.literal_eval)
all_faces_v2_df["left_eye"] = all_faces_v2_df["left_eye"].apply(ast.literal_eval)
all_faces_v2_df["nose"] = all_faces_v2_df["nose"].apply(ast.literal_eval)
all_faces_v2_df["mouth_right"] = all_faces_v2_df["mouth_right"].apply(ast.literal_eval)
all_faces_v2_df["mouth_left"] = all_faces_v2_df["mouth_left"].apply(ast.literal_eval)

In [ ]:
all_faces_v2_df['filename'] = all_faces_v2_df['bucket_url'].apply(lambda image: image.replace('../data/', ''))

In [ ]:
all_faces_v2_df['account_name'] = all_faces_v2_df['filename'].apply(lambda x: x.split('/')[0])

In [ ]:
all_faces_v2_df.head()

,image,type,bucket_url,face_uuid,retina_face_score,facial_area,right_eye,left_eye,nose,mouth_right,...,f_pixel_x,f_pixel_y,f_pixel_width,f_pixel_height,f_relativ_x,f_relativ_y,f_relativ_width,f_relativ_height,filename,account_name
0,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,1e19a944-ce54-4a50-af5f-1fb26ae46852,0.999463,"[618, 68, 698, 179]","[655.83215, 115.59756]","[687.78125, 116.61998]","[679.88385, 137.47713]","[651.548, 148.28584]",...,618,68,80,111,42.916667,7.943925,5.555556,12.967290,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet
1,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,7c50af0d-d49f-4121-921b-173b16eb9a63,0.999458,"[865, 76, 941, 176]","[882.80505, 117.19016]","[918.87573, 113.6915]","[901.9659, 135.02817]","[887.5133, 148.83932]",...,865,76,76,100,60.069444,8.878505,5.277778,11.682243,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet
2,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,f0902030-6596-4e02-a702-9ac7c47c5e16,0.999433,"[217, 96, 293, 203]","[247.93494, 139.99324]","[282.4035, 137.506]","[276.23734, 155.95622]","[253.7339, 175.65933]",...,217,96,76,107,15.069444,11.214953,5.277778,12.500000,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet
3,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,d4d4e4e0-f05f-47f5-8dc3-754335fd3e62,0.995322,"[418, 111, 460, 165]","[427.0738, 133.0951]","[445.76318, 133.36816]","[434.90997, 144.35779]","[429.42117, 153.67068]",...,418,111,42,54,29.027778,12.967290,2.916667,6.308411,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet
4,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,b57bfd65-c875-4f24-9273-ed43955386d2,0.995100,"[986, 65, 1075, 227]","[1001.7438, 125.714874]","[1003.1344, 126.80907]","[987.33704, 157.53435]","[1007.1196, 187.25154]",...,986,65,89,162,68.472222,7.593458,6.180556,18.925234,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet


## Preparing Faces

In [ ]:
#@title Align-pre detected faces

def extract_faces_adapted(uuid, align = True, resize = False):
  """
  This functions builds upon the extract_faces function from Sefik Serengils RetinaFace
  https://github.com/serengil/retinaface/blob/master/retinaface/RetinaFace.py#L58.

  extract_faces_adapted extracts and aligns (optinal) all faces in one image.

  Parameters:
      filepath (string): exact filepath of the image, where all face should be
                         extracted from

      align (boolean): toggle face alignment (default is True)

  Returns:
      Returns a list of dicts. Each dict has one key, value pair with an id and
      a corresponding extracted face image.
  """

  current_image = all_faces_v2_df.loc[all_faces_v2_df['face_uuid'] == uuid].iloc[0]

  filepath = f"/content/images/{current_image['filename']}"
  image = cv2.imread(filepath)

  facial_area = current_image['facial_area']
  facial_img = image[facial_area[1]: facial_area[3], facial_area[0]: facial_area[2]]

  if align == True:
    right_eye = current_image['right_eye']
    left_eye =  current_image['left_eye']
    nose = current_image['nose']
    facial_img = postprocess.alignment_procedure(facial_img, right_eye, left_eye, nose)

  image_array = facial_img[0]  # Accessing the numpy array part of the tuple

  file_name = f'/content/Exports/{uuid}.jpg'
  return cv2.imwrite(file_name, image_array[:, :, ::1])

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas()

if os.path.exists('/content/Exports') == False:
  os.mkdir('/content/Exports')


all_faces_v2_df['face_uuid'].progress_apply(extract_faces_adapted)

  0%|          | 0/5210 [00:00<?, ?it/s]

0       True
1       True
2       True
3       True
4       True
        ... 
5205    True
5206    True
5207    True
5208    True
5209    True
Name: face_uuid, Length: 5210, dtype: bool

In [ ]:
!zip -r ../data/2024-04-17-Faces-Posts-Images.zip Exports

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
  adding: Exports/906412f9-80a3-4baa-bc1b-8c00da40ec11.jpg (deflated 18%)
  adding: Exports/b82017b0-c82c-477a-8ba9-987c8ef375a3.jpg (deflated 1%)
  adding: Exports/23ab169e-bed1-489f-bd1b-487ee97fdb3f.jpg (deflated 15%)
  adding: Exports/07752f44-9c88-4148-a4c1-df4022c2b2bb.jpg (deflated 2%)
  adding: Exports/305181e7-ec56-4ada-80cd-578bbe55b3e8.jpg (deflated 14%)
  adding: Exports/29184bdc-eca9-4712-9f3e-bd14cc2a6653.jpg (deflated 8%)
  adding: Exports/7cf69cd0-7384-479d-9c0e-f89a59092f8e.jpg (deflated 18%)
  adding: Exports/79cbaf06-aeff-4582-8bb4-24ebefba66df.jpg (deflated 4%)
  adding: Exports/5552551c-70ce-493c-9023-aba4e3cf4025.jpg (deflated 0%)
  adding: Exports/a91ff2a2-ccf9-4e79-b570-1c8d5ec8c2dd.jpg (deflated 16%)
  adding: Exports/7de29690-0bb1-4bc7-870d-f91e71082c35.jpg (deflated 8%)
  adding: Exports/9ef26da7-208a-440e-a9d6-d7155fa8da03.jpg (deflated 8%)
  adding: Exports/10c980ba-f5d1-4332-b82b-aed7e65fbc

## Preparing Face Database
Hier habe ich mich nun auf die Kanzlerkandidaten konzentriert.

QuickFix um mit Umlauten gut klar zu kommen.

In [ ]:
import os
import unicodedata

DATABASE = "/content/database_final-v1/2024-01-03-database-final-v4"


def replace_characters(original):
    normalized = unicodedata.normalize('NFC', original)
    # Replace specific characters in the string
    return normalized.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue')

def rename_files_and_folders(path):
    for root, dirs, files in os.walk(path, topdown=False):
        # Rename files
        for name in files:
            new_name = replace_characters(name)
            if new_name != name:
                os.rename(os.path.join(root, name), os.path.join(root, new_name))

        # Rename directories
        for name in dirs:
            new_name = replace_characters(name)
            if new_name != name:
                os.rename(os.path.join(root, name), os.path.join(root, new_name))

rename_files_and_folders(DATABASE)

Filtere nach den Accounts, die mich interessieren

In [ ]:
# Filtere für Kanzlerkandidaten
selected_accounts = ['armin_laschet', 'abaerbock', 'christlichsozialeunion',
       'christianlindner', 'cdu', 'fdp', 'die_gruenen', 'markus.soeder',
       'spdde', 'olafscholz']

all_faces_v2_df = all_faces_v2_df[all_faces_v2_df['account_name'].isin(selected_accounts)]

In [ ]:
all_faces_v2_df.head()

,image,type,bucket_url,face_uuid,retina_face_score,facial_area,right_eye,left_eye,nose,mouth_right,...,f_pixel_y,f_pixel_width,f_pixel_height,f_relativ_x,f_relativ_y,f_relativ_width,f_relativ_height,filename,account_name,Party
0,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,1e19a944-ce54-4a50-af5f-1fb26ae46852,0.999463,"[618, 68, 698, 179]","[655.83215, 115.59756]","[687.78125, 116.61998]","[679.88385, 137.47713]","[651.548, 148.28584]",...,68,80,111,42.916667,7.943925,5.555556,12.967290,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
1,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,7c50af0d-d49f-4121-921b-173b16eb9a63,0.999458,"[865, 76, 941, 176]","[882.80505, 117.19016]","[918.87573, 113.6915]","[901.9659, 135.02817]","[887.5133, 148.83932]",...,76,76,100,60.069444,8.878505,5.277778,11.682243,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
2,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,f0902030-6596-4e02-a702-9ac7c47c5e16,0.999433,"[217, 96, 293, 203]","[247.93494, 139.99324]","[282.4035, 137.506]","[276.23734, 155.95622]","[253.7339, 175.65933]",...,96,76,107,15.069444,11.214953,5.277778,12.500000,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
3,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,d4d4e4e0-f05f-47f5-8dc3-754335fd3e62,0.995322,"[418, 111, 460, 165]","[427.0738, 133.0951]","[445.76318, 133.36816]","[434.90997, 144.35779]","[429.42117, 153.67068]",...,111,42,54,29.027778,12.967290,2.916667,6.308411,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
4,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,b57bfd65-c875-4f24-9273-ed43955386d2,0.995100,"[986, 65, 1075, 227]","[1001.7438, 125.714874]","[1003.1344, 126.80907]","[987.33704, 157.53435]","[1007.1196, 187.25154]",...,65,89,162,68.472222,7.593458,6.180556,18.925234,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU


Mapping Account <-> Partei

In [ ]:
party_dict = {
  "armin_laschet": "CDU",
  "spdde": "SPD",
  "afd.bund": "AfD",
  "nicola_beer": "FDP",
  "christianlindner": "FDP",
  "abaerbock": "GRUEN",
  "cdu": "CDU",
  "robert.habeck":  "GRUEN",
  "olafscholz": "SPD",
  "fw_bayern": "FW",
  "die_gruenen":  "GRUEN",
  "christlichsozialeunion":  "CSU",
  "fdp": "FDP",
  "saskiaesken":  "SPD",
  "susanne_hennig_wellsow": "Linke",
  "atesgurpinar":  "Linke",
  "grey_gor": "FW",
  "markus.soeder":  "CSU",
  "dielinke":  "Linke",
  "joerg.meuthen": "AfD",
  "engin_eroglu_": "FW",
}
all_faces_v2_df['Party'] = all_faces_v2_df['account_name'].map(party_dict)

In [ ]:
all_faces_v2_df.head()

,image,type,bucket_url,face_uuid,retina_face_score,facial_area,right_eye,left_eye,nose,mouth_right,...,f_pixel_y,f_pixel_width,f_pixel_height,f_relativ_x,f_relativ_y,f_relativ_width,f_relativ_height,filename,account_name,Party
0,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,1e19a944-ce54-4a50-af5f-1fb26ae46852,0.999463,"[618, 68, 698, 179]","[655.83215, 115.59756]","[687.78125, 116.61998]","[679.88385, 137.47713]","[651.548, 148.28584]",...,68,80,111,42.916667,7.943925,5.555556,12.967290,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
1,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,7c50af0d-d49f-4121-921b-173b16eb9a63,0.999458,"[865, 76, 941, 176]","[882.80505, 117.19016]","[918.87573, 113.6915]","[901.9659, 135.02817]","[887.5133, 148.83932]",...,76,76,100,60.069444,8.878505,5.277778,11.682243,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
2,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,f0902030-6596-4e02-a702-9ac7c47c5e16,0.999433,"[217, 96, 293, 203]","[247.93494, 139.99324]","[282.4035, 137.506]","[276.23734, 155.95622]","[253.7339, 175.65933]",...,96,76,107,15.069444,11.214953,5.277778,12.500000,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
3,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,d4d4e4e0-f05f-47f5-8dc3-754335fd3e62,0.995322,"[418, 111, 460, 165]","[427.0738, 133.0951]","[445.76318, 133.36816]","[434.90997, 144.35779]","[429.42117, 153.67068]",...,111,42,54,29.027778,12.967290,2.916667,6.308411,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU
4,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,b57bfd65-c875-4f24-9273-ed43955386d2,0.995100,"[986, 65, 1075, 227]","[1001.7438, 125.714874]","[1003.1344, 126.80907]","[987.33704, 157.53435]","[1007.1196, 187.25154]",...,65,89,162,68.472222,7.593458,6.180556,18.925234,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet,CDU


Schiebe die Bilder in einen neuen Unterordner ( `Datenbank > Partei > Name `)

In [ ]:
import os
import shutil

accounts_dict = {
  "CDU": ["Armin Laschet"],
  "CSU": ["Markus Soeder"],
  "SPD": ["Olaf Scholz"],
  "FDP": ["Christian Lindner"],
  "GRUEN": ["Annalena Baerbock"],
}

def copy_directory(src, dst):
    if os.path.exists(dst):
        os.rename(dst, dst + '_backup')
    shutil.copytree(src, dst)

# New base directory for the structured data
new_base_dir = "database_v2"

# Iterate through parties and their members in the dictionary
for party, names in accounts_dict.items():
    for name in names:
        # Source and destination directory paths
        src_dir = os.path.join(DATABASE, name)
        dst_dir = os.path.join(new_base_dir, party, name)

        # Copy directory if it exists in the source
        if os.path.isdir(src_dir):
            copy_directory(src_dir, dst_dir)
        else:
            print(f"Directory not found: {src_dir}")

## Starte die Gesichtserkennung.
Die Datenbank wird beim ersten Mal für jede Partei generiert. Das erhöht die Geschwindigkeit.

In [ ]:
#@title Perform Face Recognition

DATABASE = "/content/database_v2"
EXPORT_FOLDER = "../data/" #@param {"type": "string"}
VERSION = 0 # @param {type:"number"}
FORCE_DETECT = False
BACKEND = "retinaface" #@param ['skip', 'opencv', 'ssd', 'dlib','mtcnn', 'retinaface', 'mediapipe', 'yolov8',' yunet']

def find_matches(image, model, metric, account_name):
    # Function to find matches for a given image using specified model and metric
    path = f'/content/Exports/{image}.jpg'
    col =  "distance" # f"{model}_{metric}"
    db_path = os.path.join(DATABASE, account_name)  # Updated db_path
    scores = DeepFace.find(img_path=path, db_path=db_path, model_name=model,
                           distance_metric=metric, detector_backend=BACKEND,
                           enforce_detection=FORCE_DETECT, silent=True)

    return scores, col


def process_scores(scores, col, image):
    # Process and return match details
    face_score, match_file, matches = np.nan, "", []
    if len(scores[0]) > 0:
          scores_df = scores[0]
          min_score = scores_df[scores_df[col] == scores_df[col].min()]
          face_score = min_score[col].iloc[0]
          match_file = min_score['identity'].iloc[0].split("/")[5]
          matches = [{"match_uuid": f'match_{uuid.uuid4()}', "face_uuid": image,
                      "match_file": row['identity'].split("/")[5],
                      "match_score": row[col], "model": model,
                      "distance_metric": metric}
                    for index, row in scores_df.iterrows()]

    return face_score, match_file, matches

# Main loop
model = "ArcFace"
metrics = ['euclidean_l2'] #'euclidean', 'cosine']

for metric in tqdm(metrics, desc='Processing Metrics'):
    faces, all_matches = [], []

    # Group by 'account_name' and iterate through each group
    for party, group in tqdm(all_faces_v2_df.groupby('Party'), desc='Processing parties'):
        for image in tqdm(group['face_uuid']):
            scores, col = find_matches(image, model, metric, party)
            face_score, match_file, matches = process_scores(scores, col, image)
            all_matches.extend(matches)
            if face_score is not np.nan:
                faces.append({"face_uuid": image, "match_file": match_file,
                              "match_score": face_score, "model": model,
                              "distance_metric": metric, "number_of_hits": len(scores[0])})

    # Creating and Saving DataFrames for each model-metric pair
    all_matches_df = pd.DataFrame.from_dict(all_matches)
    face_recognition_df = pd.DataFrame.from_dict(faces)

    all_matches_df.to_csv(f"{EXPORT_FOLDER}all_matches_{model}_{metric}_{VERSION}.csv", index=False)
    face_recognition_df.to_csv(f"{EXPORT_FOLDER}top_matches_{model}_{metric}_{VERSION}.csv", index=False)

Processing Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

Processing parties:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/1269 [00:00<?, ?it/s]

24-04-18 10:06:56 - arcface_weights.h5 will be downloaded to /root/.deepface/weights/arcface_weights.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/arcface_weights.h5
To: /root/.deepface/weights/arcface_weights.h5



  0%|          | 0.00/137M [00:00<?, ?B/s]


  2%|▏         | 2.10M/137M [00:00<00:07, 19.2MB/s]


  6%|▌         | 8.39M/137M [00:00<00:02, 43.3MB/s]


 13%|█▎        | 17.3M/137M [00:00<00:02, 56.5MB/s]


 17%|█▋        | 23.1M/137M [00:00<00:02, 56.8MB/s]


 21%|██▏       | 29.4M/137M [00:00<00:01, 58.1MB/s]


 26%|██▌       | 35.7M/137M [00:00<00:01, 50.8MB/s]


 30%|███       | 41.4M/137M [00:00<00:01, 52.8MB/s]


 34%|███▍      | 47.2M/137M [00:00<00:01, 47.5MB/s]


 39%|███▉      | 53.5M/137M [00:01<00:01, 51.1MB/s]


 44%|████▎     | 59.8M/137M [00:01<00:01, 47.5MB/s]


 48%|████▊     | 65.5M/137M [00:01<00:01, 50.0MB/s]


 52%|█████▏    | 71.8M/137M [00:01<00:01, 53.1MB/s]


 57%|█████▋    | 77.6M/137M [00:01<00:01, 47.9MB/s]


 61%|██████    | 83.9M/137M [00:01<00:01, 51.3MB/s]


 66%|██████▌   | 90.2M/137M [00:01<00:00, 

  0%|          | 0/551 [00:00<?, ?it/s]

  0%|          | 0/957 [00:00<?, ?it/s]

  0%|          | 0/542 [00:00<?, ?it/s]

  0%|          | 0/560 [00:00<?, ?it/s]